In [2]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
!pip install transformers pillow

Looking in indexes: https://download.pytorch.org/whl/cpu

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [4]:
import os  # work with files/folders
import shutil  # copy files
import numpy as np  # numerical arrays
import pandas as pd  # tables/dataframes
import torch  # PyTorch
from PIL import Image  # image loading/processing

from huggingface_hub import hf_hub_download  # HF file downloader (used only if local file missing)

# Optional (for patch scoring like the official example)
try:  # try to import scikit-image functions
    from skimage.feature import graycomatrix, graycoprops  # GLCM features
    SKIMAGE_OK = True  # flag: scikit-image is available
except Exception:  # if scikit-image isn't installed
    SKIMAGE_OK = False  # flag: scikit-image not available


CLASSES = [  # output class names in the same order as the model output
    "authentic",
    "dalle-3-images",
    "diffusiondb",
    "midjourney-images",
    "midjourney_tti",
    "realisticSDXL",
]

LOCAL_MODEL_DIR = "./models/susy"  # local folder to store the model file


def _safe_open_rgb(image_path: str) -> Image.Image:
    img = Image.open(image_path)  # open image from disk
    if img.mode != "RGB":  # check color mode
        img = img.convert("RGB")  # convert to RGB if needed
    return img  # return PIL image


def _extract_patches(image: Image.Image, patch_size: int) -> np.ndarray:
    """Return patches as uint8 array: (N, H, W, 3). Uses non-overlapping grid like HF example."""
    w, h = image.size  # get image width and height
    num_x = w // patch_size  # how many patches fit in x direction
    num_y = h // patch_size  # how many patches fit in y direction

    if num_x == 0 or num_y == 0:  # if image is smaller than one patch
        new_w = max(w, patch_size)  # new width at least patch_size
        new_h = max(h, patch_size)  # new height at least patch_size
        print(f"[WARN] Image smaller than {patch_size}px. Resizing to {new_w}x{new_h}.")  # log resize
        image = image.resize((new_w, new_h))  # resize image
        w, h = image.size  # update size
        num_x = w // patch_size  # recompute patches in x
        num_y = h // patch_size  # recompute patches in y

    patches = np.zeros((num_x * num_y, patch_size, patch_size, 3), dtype=np.uint8)  # allocate patch array

    idx = 0  # patch counter
    for i in range(num_x):  # loop over patch columns
        for j in range(num_y):  # loop over patch rows
            x = i * patch_size  # top-left x of patch
            y = j * patch_size  # top-left y of patch
            patch = image.crop((x, y, x + patch_size, y + patch_size))  # crop patch
            patches[idx] = np.array(patch, dtype=np.uint8)  # store patch as uint8
            idx += 1  # increment patch counter

    return patches  # return patches array


def _glcm_contrast_score(patch_uint8_hwc: np.ndarray) -> float:
    """Compute GLCM contrast score (similar to official example)."""
    patch_img = Image.fromarray(patch_uint8_hwc)  # convert array to PIL image
    gray = patch_img.convert("L")  # convert to grayscale
    gray_np = np.array(gray, dtype=np.uint8)  # grayscale as uint8 array

    glcm = graycomatrix(  # compute GLCM matrix
        gray_np,  # grayscale image
        distances=[5],  # distance used in GLCM
        angles=[0],  # angle used in GLCM
        levels=256,  # gray levels
        symmetric=True,  # symmetric matrix
        normed=True,  # normalized matrix
    )
    return float(graycoprops(glcm, "contrast")[0, 0])  # return contrast feature


def detect_synthetic_susy(
    image_path: str,
    model_repo: str = "HPAI-BSC/SuSy",
    model_filename: str = "SuSy.pt",
    patch_size: int = 224,
    top_k_patches: int = 5,
    threshold: float = 0.5,
    use_glcm_topk: bool = True,
):
    print("========================================")  # separator
    print("[1] Starting synthetic detection (SuSy)")  # step log
    print(f"[1] Image: {image_path}")  # log image path
    print(f"[1] HF repo: {model_repo}")  # log repo id
    print("========================================")  # separator

    if not os.path.exists(image_path):  # check image exists
        raise FileNotFoundError(f"Image not found: {image_path}")  # raise error if missing

    device = "cuda" if torch.cuda.is_available() else "cpu"  # choose device
    print(f"[2] Using device: {device}")  # log device

    os.makedirs(LOCAL_MODEL_DIR, exist_ok=True)  # ensure local model folder exists
    local_model_path = os.path.join(LOCAL_MODEL_DIR, model_filename)  # local path to model file

    print("[3] Loading model from local directory (download only if missing)...")  # step log
    if os.path.exists(local_model_path):  # if model exists locally
        model_path = local_model_path  # use local file
        print(f"[3] Found local model: {model_path}")  # log local model path
    else:  # if model not found locally
        print(f"[3] Local model not found at: {local_model_path}")  # log missing file
        print("[3] Downloading model from Hugging Face to cache...")  # log download
        cached_path = hf_hub_download(repo_id=model_repo, filename=model_filename)  # download to HF cache
        print(f"[3] HF cached path: {cached_path}")  # log cached location
        shutil.copy2(cached_path, local_model_path)  # copy from cache to your local folder
        model_path = local_model_path  # now use local file
        print(f"[3] Copied model to local path: {model_path}")  # log local save

    print("[4] Loading TorchScript model...")  # step log
    model = torch.jit.load(model_path, map_location=device)  # load TorchScript model
    model.eval()  # set eval mode
    print("[4] Model loaded and set to eval mode.")  # log success

    print("[5] Loading image...")  # step log
    image = _safe_open_rgb(image_path)  # open image as RGB
    print(f"[5] Image mode: {image.mode}, size: {image.size}")  # log image info

    print(f"[6] Extracting non-overlapping {patch_size}x{patch_size} patches...")  # step log
    patches = _extract_patches(image, patch_size=patch_size)  # extract patches
    print(f"[6] Total patches extracted: {len(patches)}")  # log patch count

    if len(patches) == 0:  # safety check
        raise RuntimeError("No patches could be extracted from the image.")  # raise error

    chosen = patches  # default: use all patches
    if use_glcm_topk and SKIMAGE_OK:  # if using GLCM and scikit-image is available
        print("[7] Scoring patches with GLCM contrast (top-K selection)...")  # step log
        scores = []  # list for patch scores
        for i, p in enumerate(patches):  # loop patches
            s = _glcm_contrast_score(p)  # compute contrast score
            scores.append(s)  # store score
            if i < 3:  # print first few scores
                print(f"    [7] Patch {i:02d} contrast score: {s:.4f}")  # log score

        scores = np.array(scores, dtype=np.float32)  # convert scores to numpy array
        order = np.argsort(scores)[::-1]  # sort indices by descending score

        k = min(top_k_patches, len(patches))  # choose K not bigger than available patches
        print(f"[7] Selecting top {k} patches (requested top_k={top_k_patches}).")  # log selection
        chosen = patches[order[:k]]  # select top-K patches

        print("[7] Top patch indices & scores:")  # log header
        for rank, idx in enumerate(order[:k]):  # loop selected indices
            print(f"    rank {rank + 1}: patch_idx={int(idx)}, contrast={float(scores[idx]):.4f}")  # log each
    else:  # if not using GLCM top-K
        if use_glcm_topk and not SKIMAGE_OK:  # user wanted GLCM but scikit-image missing
            print("[7] WARNING: scikit-image not available. Skipping GLCM top-K; using all patches.")  # warn
        else:  # user disabled GLCM top-K
            print("[7] Using all patches (no top-K selection).")  # log

    print("[8] Converting patches to tensor...")  # step log
    x = torch.from_numpy(np.transpose(chosen, (0, 3, 1, 2))).float() / 255.0  # HWC->CHW and scale to [0,1]
    x = x.to(device)  # move to cpu/cuda
    print(  # log tensor info
        f"[8] Tensor shape: {tuple(x.shape)}, dtype={x.dtype}, min={x.min().item():.4f}, max={x.max().item():.4f}"
    )

    print("[9] Running inference...")  # step log
    with torch.no_grad():  # disable gradients
        preds = model(x)  # run model (N,6) probabilities
    preds_cpu = preds.detach().cpu().numpy()  # move to CPU numpy

    df = pd.DataFrame(preds_cpu, columns=CLASSES)  # build dataframe with class names
    print("[9] Per-patch probabilities (first 10 rows):")  # log header
    print(df.head(10).to_string(index=False))  # print first 10 rows

    print("[10] Aggregating across patches (mean probs)...")  # step log
    mean_probs = df.mean(axis=0)  # mean probability per class
    mean_dict = {k: float(mean_probs[k]) for k in CLASSES}  # convert to python floats
    print("[10] Mean probabilities:")  # log header
    for k in CLASSES:  # loop classes
        print(f"    {k:16s}: {mean_dict[k]:.4f}")  # print mean prob

    synthetic_prob = 1.0 - mean_dict["authentic"]  # synthetic score
    decision = "SYNTHETIC" if synthetic_prob >= threshold else "AUTHENTIC"  # threshold decision
    print("[11] Final decision:")  # log header
    print(f"    synthetic_prob = {synthetic_prob:.4f} (threshold={threshold:.2f})")  # print score
    print(f"    ==> {decision}")  # print decision

    return {  # return results as dict
        "image_path": image_path,  # original image path
        "model_repo": model_repo,  # HF repo id
        "patch_size": patch_size,  # patch size used
        "num_patches_used": int(len(chosen)),  # number of patches used
        "mean_probs": mean_dict,  # mean probabilities
        "synthetic_prob": float(synthetic_prob),  # synthetic probability
        "threshold": float(threshold),  # threshold used
        "decision": decision,  # final decision
    }


if __name__ == "__main__":
    result = detect_synthetic_susy("image.jpg", top_k_patches=5, threshold=0.5, use_glcm_topk=True)  # run detection
    print("\nReturned result dict:")  # print header
    print(result)  # print returned dict

[1] Starting synthetic detection (SuSy)
[1] Image: image.jpg
[1] HF repo: HPAI-BSC/SuSy
[2] Using device: cuda
[3] Loading model from local directory (download only if missing)...
[3] Found local model: ./models/susy/SuSy.pt
[4] Loading TorchScript model...
[4] Model loaded and set to eval mode.
[5] Loading image...
[5] Image mode: RGB, size: (928, 1120)
[6] Extracting non-overlapping 224x224 patches...
[6] Total patches extracted: 20
[7] WARNING: scikit-image not available. Skipping GLCM top-K; using all patches.
[8] Converting patches to tensor...
[8] Tensor shape: (20, 3, 224, 224), dtype=torch.float32, min=0.0000, max=1.0000
[9] Running inference...
[9] Per-patch probabilities (first 10 rows):
 authentic  dalle-3-images  diffusiondb  midjourney-images  midjourney_tti  realisticSDXL
  0.219229        0.564878     0.013112           0.087195        0.025122       0.090464
  0.109307        0.649077     0.012508           0.109269        0.021345       0.098494
  0.005084        0.899